# Assumptions and Observations

## Observations

### Production runs
- Runs never overlap on a machine
- R104 has no end time, can't assume if still running or glitched

### Counter readings
- C009 -> C010 drop from 1760 to 25
- C021 -> C022 drop from 1020 to 1018
- `PKG-99` doesn't have any associated `production_run`

### Downtime events
- Reasons are all over the place
- 5 of 10 events say `Unknown`
- `D003` and `D004` overlap on `PKG-01`, how can one downtime event appear when there is already downtime?
- `D002` starts 2 minutes before `R101`
- Curious if operator stop is meant to end a `production_run` and also include end time for that run

## Decisions and assumptions

### Ground rules
- Counts are a running total, so they should only go up.
- A reading belongs to a run on the same machine if its time is between the run's start and end, including exactly at the start or end.
- A stop belongs to a run on the same machine if any part of it overlaps the run.
- All times are in UTC, and the time zone offsets in the data are assumed to be correct.
- Target rates in cases/minute are multiplied by 60 to get cases/hour.

### Decisions that change the result

#### 1. `C010` drops from 1760 to 25 in the middle of `R101`
- **Decision:** treat it as a glitch and ignore it. `C011` (210) is ignored too, because it's still below 1760.
- **Why:** a running total shouldn't go down.
- **Effect:** `R101` is only counted up to 14:40: 250 cases, 49.5% of target.
- **Other option:** treat it as a counter reset: 460 cases, 91.1%.

#### 2. `C001` and `C014` come 1 minute before `R100` and `R102` start
- **Decision:** leave them out.
- **Why:** they're outside the run.
- **Effect:** `R100` is counted from 13:10 (420 cases, 78%) and `R102` from 13:30 (280 cases, 73%).
- **Other option:** use them as the starting count: 510 cases (94%) and 370 cases (96%).

#### 3. `D002` starts 2 minutes before `R101`
- **Decision:** count only its 4 minutes inside `R101`, not its full 6.
- **Why:** only the part inside the run is downtime for that run.
- **Effect:** `R101` has 9.5 minutes of downtime.
- **Other option:** leave `D002` out: 5.5 minutes.

#### 4. `D003` and `D004` overlap by 1.5 minutes
- **Decision:** count the shared time once, 5.5 minutes in total.
- **Why:** the same downtime shouldn't count twice.
- **Other option:** drop `D004`: 4.0 minutes.

#### 5. `R104` has no end time
- **Decision:** count `R104`'s cases, but give it no throughput or attainment.
- **Why:** `C012` and `C013` are inside the run and the count goes up, so they're good readings. The missing end time and the operator stop (`D010` at 15:34) don't change what the counter read: a running total just stops rising while the machine is down. Throughput needs operating time, though, and that needs an end. `D010` has no end either, so the run may still be going or still be stopped.
- **Effect:** `R104` has 150 cases by 15:40, a minimum. It isn't ranked, and `D010` adds no downtime.
- **Other option:** use 15:40, the last reading, as its end: 83% with `D010` left out, or 119% with `D010` counted to 15:40.

### Left out, with no effect on the result
- `C022` (1020 → 1018) is a glitch. It can't be a reset: getting back to 1018 within 15 minutes would mean about 4,000 cases/hour against a 420 target.
- `C004` is a duplicate of `C003` (same machine, time and count).
- `C005` says `"ERROR"`. No cases are lost, because counts are a running total.
- `C024` is on `PKG-99`, which has no production runs.

### Result
`R101` is the lowest-performing completed run at 49.5% of target, mainly because of decision 1. The others: `R100` 77.8%, `R102` 72.7%, `R103` 73.4%.

## What the data can't tell us
- Which run performed worst depends on the rules above:
  - `R101` (49.5%) with these rules
  - `R102` (72.7%) if the `C010` drop is a reset
  - `R103` (73.4%) if the `C010` drop is a reset and `C001`/`C014` are used as starting counts
- `R104`'s throughput and attainment can't be measured because it has no end time. Only its 150 cases so far are known.
- The data doesn't record why counts dropped, and 5 of 10 stops have the reason `Unknown`.

## Matching readings and stops to runs
Times are UTC.
- A **counter reading** belongs to a run on the same machine if its time falls between the run's start and end. A reading exactly at the start or end counts.
- A **downtime stop** belongs to a run on the same machine if any part of it overlaps the run. Only the overlapping minutes count, and time shared by two stops counts once.
- Readings and stops that can't be used are listed on their run's row, with the reason. Readings that don't match any run are listed above the table.

In [1]:
%run 02_cleaning_data.ipynb

from datetime import datetime


def ts(timestamp):
    return None if timestamp is None else datetime.fromisoformat(timestamp)


def hhmm(timestamp):
    # "13:30", or "14:43:30" when the seconds aren't zero
    if not timestamp:
        return "no end"
    return timestamp[11:16] if timestamp[17:19] == "00" else timestamp[11:19]


def minutes(start, end):
    return (end - start).total_seconds() / 60


def run_for_reading(reading):
    at = ts(reading["recorded_at"])
    for run in production_runs:
        if (run["equipment_id"] == reading["equipment_id"] and ts(run["start_time"]) <= at
                and (run["end_time"] is None or at <= ts(run["end_time"]))):
            return run
    return None


def run_for_stop(stop):
    start, end = ts(stop["start_time"]), ts(stop["end_time"])
    for run in production_runs:
        if (run["equipment_id"] == stop["equipment_id"] and (end is None or end > ts(run["start_time"]))
                and (run["end_time"] is None or start < ts(run["end_time"]))):
            return run
    return None


run_machines = {run["equipment_id"] for run in production_runs}

# Counter readings: match each to a run, then check whether it can be used
readings, seen, last_good = [], set(), {}
for r in sorted(counter_readings, key=lambda r: (r["equipment_id"], r["recorded_at"])):
    run = run_for_reading(r)
    key = (r["equipment_id"], r["recorded_at"], r["count"])
    if r["equipment_id"] not in run_machines:
        why_not = "machine has no production runs"
    elif run is None:
        why_not = "outside every run"
    elif not isinstance(r["count"], int):
        why_not = "count isn't a number"
    elif key in seen:
        why_not = "duplicate"
    elif run["run_id"] in last_good and r["count"] < last_good[run["run_id"]]:
        why_not = "glitch: count dropped"
    else:
        why_not = ""
        last_good[run["run_id"]] = r["count"]
    seen.add(key)
    readings.append({"reading_id": r["reading_id"], "equipment_id": r["equipment_id"], "time": hhmm(r["recorded_at"]),
                     "count": r["count"], "run_id": run["run_id"] if run else "",
                     "used": "no" if why_not else "yes", "why_not": why_not})

# Downtime events: match each to the run it overlaps, and count only the overlapping minutes.
# Time already counted for an earlier stop in the same run isn't counted again.
events, counted_until = [], {}
for e in sorted(downtime_events, key=lambda e: (e["equipment_id"], e["start_time"])):
    run = run_for_stop(e)
    row = {"event_id": e["event_id"], "equipment_id": e["equipment_id"],
           "time": f'{hhmm(e["start_time"])}–{hhmm(e["end_time"])}', "reason": e["reason"],
           "run_id": run["run_id"] if run else "", "minutes_counted": None, "used": "no", "note": ""}
    if run is None:
        row["note"] = "outside every run"
    elif run["end_time"] is None:
        row["note"] = "run has no end time"
    else:
        start, end = ts(e["start_time"]), ts(e["end_time"])
        run_start, run_end = ts(run["start_time"]), ts(run["end_time"])
        inside_from, inside_to = max(start, run_start), min(end or run_end, run_end)
        count_from = max(inside_from, counted_until.get(run["run_id"], run_start))
        row["minutes_counted"] = max(0.0, minutes(count_from, inside_to))
        counted_until[run["run_id"]] = max(inside_to, counted_until.get(run["run_id"], run_start))
        row["used"] = "yes"
        if start < run_start:
            row["note"] = f"starts before {run['run_id']}: only the overlap counts"
        elif row["minutes_counted"] < minutes(inside_from, inside_to):
            row["note"] = "shares time with an earlier stop: counted once"
    events.append(row)

# One row per production run, with the readings and stops associated with it
def describe_stop(e):
    short = {"starts before": "overlap only", "shares time": "shared time counted once"}
    note = next((text for start, text in short.items() if e["note"].startswith(start)), "")
    return f'{e["event_id"]} {e["minutes_counted"]:.1f} min' + (f" ({note})" if note else "")


run_table = []
for run in production_runs:
    matched_readings = [r for r in readings if r["run_id"] == run["run_id"]]
    matched_stops = [e for e in events if e["run_id"] == run["run_id"]]
    used = [r for r in matched_readings if r["used"] == "yes"]
    stops_used = [e for e in matched_stops if e["used"] == "yes"]
    no_end = run["end_time"] is None
    run_table.append({
        "run_id": run["run_id"],
        "equipment_id": run["equipment_id"],
        "product_code": run["product_code"],
        "window": f'{hhmm(run["start_time"])}–{hhmm(run["end_time"])}',
        "target_hourly_rate": run["target_hourly_rate"],
        "counter_readings_used": ", ".join(r["reading_id"] for r in used),
        "counter_readings_not_used": ", ".join(f'{r["reading_id"]} ({r["why_not"]})'
                                               for r in matched_readings if r["used"] == "no"),
        "cases_produced": used[-1]["count"] - used[0]["count"],
        "downtime_events_used": ", ".join(describe_stop(e) for e in stops_used),
        "downtime_events_not_used": ", ".join(f'{e["event_id"]} ({e["note"]})'
                                              for e in matched_stops if e["used"] == "no"),
        "downtime_minutes": None if no_end else sum(e["minutes_counted"] for e in stops_used),
    })

unmatched = ([f'{r["reading_id"]} ({r["why_not"]})' for r in readings if not r["run_id"]]
             + [f'{e["event_id"]} ({e["note"]})' for e in events if not e["run_id"]])

### Production runs

In [2]:
print("Not matched to any run:", ", ".join(unmatched) or "none")
pd.DataFrame(run_table).astype({"cases_produced": "Int64"}).style.hide(axis="index").format(precision=1, na_rep="")

Not matched to any run: C001 (outside every run), C014 (outside every run), C024 (machine has no production runs)


run_id,equipment_id,product_code,window,target_hourly_rate,counter_readings_used,counter_readings_not_used,cases_produced,downtime_events_used,downtime_events_not_used,downtime_minutes
R100,PKG-01,FLOUR-A,13:00–14:00,600,"C002, C003, C006","C004 (duplicate), C005 (count isn't a number)",420,D001 6.0 min,,6.0
R101,PKG-01,FLOUR-A,14:10–15:10,600,"C007, C008, C009","C010 (glitch: count dropped), C011 (glitch: count dropped)",250,"D002 4.0 min (overlap only), D003 4.0 min, D004 1.5 min (shared time counted once)",,9.5
R102,PKG-02,FLOUR-B,13:15–14:15,420,"C015, C016, C017, C018",,280,D005 5.0 min,,5.0
R103,PKG-02,FLOUR-B,14:30–15:30,420,"C019, C020, C021, C023",C022 (glitch: count dropped),290,"D006 1.0 min, D007 0.8 min, D008 0.9 min, D009 0.8 min",,3.6
R104,PKG-01,FLOUR-C,15:20–no end,540,"C012, C013",,150,,D010 (run has no end time),
